In [8]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

In [11]:
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [28]:
url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
	"latitude": [-6.1959, -6.2398, -6.9172, -7.8012, -7.2587],
	"longitude": [106.827, 106.9757, 107.6198, 110.3605, 112.7535],
	"hourly": ["pm10", "pm2_5", "carbon_monoxide", "carbon_dioxide", "dust", "uv_index", "nitrogen_dioxide"],
	"current": ["carbon_monoxide", "dust", "uv_index", "pm2_5", "pm10", "nitrogen_dioxide", "us_aqi"],
}
responses = openmeteo.weather_api(url, params = params)

In [29]:
city_names = ["Jakarta Kota", "Bekasi", "Bandung", "Yogyakarta", "Surabaya"]

all_city_data = []

In [30]:
for i, response in enumerate(responses):
    city = city_names[i]

    hourly = response.Hourly()

    start_time = pd.to_datetime(hourly.Time(), unit="s", utc=True).tz_convert("Asia/Jakarta")
    end_time = pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True).tz_convert("Asia/Jakarta")
    interval = pd.Timedelta(seconds=hourly.Interval())

    data_range = pd.date_range(start=start_time, end=end_time, freq=interval, inclusive="left")

    city_df = pd.DataFrame({
        "time": data_range,
        "city_name": city,
        "pm10": hourly.Variables(0).ValuesAsNumpy(),
        "pm2_5": hourly.Variables(1).ValuesAsNumpy(),
        "carbon_monoxide": hourly.Variables(2).ValuesAsNumpy(),
        "carbon_dioxide": hourly.Variables(3).ValuesAsNumpy(),
        "dust": hourly.Variables(4).ValuesAsNumpy(),
        "uv_index": hourly.Variables(5).ValuesAsNumpy(),
        "nitrogen_dioxide": hourly.Variables(6).ValuesAsNumpy()
    })

    all_city_data.append(city_df)

df_air_quality = pd.concat(all_city_data, ignore_index=True)

df_air_quality.head()

,time,city_name,pm10,pm2_5,carbon_monoxide,carbon_dioxide,dust,uv_index,nitrogen_dioxide
0,2026-07-27 07:00:00+07:00,Jakarta Kota,72.599998,68.500000,2431.0,497.0,0.0,0.4,57.000000
1,2026-07-27 08:00:00+07:00,Jakarta Kota,79.199997,75.000000,2280.0,489.0,0.0,1.6,52.000000
2,2026-07-27 09:00:00+07:00,Jakarta Kota,58.200001,54.000000,1797.0,473.0,0.0,3.8,41.799999
3,2026-07-27 10:00:00+07:00,Jakarta Kota,44.799999,41.200001,1360.0,459.0,0.0,6.3,31.799999
4,2026-07-27 11:00:00+07:00,Jakarta Kota,38.299999,35.599998,1087.0,453.0,0.0,8.1,23.000000


In [32]:
df_air_quality.info()
df_air_quality.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype                       
---  ------            --------------  -----                       
 0   time              600 non-null    datetime64[ns, Asia/Jakarta]
 1   city_name         600 non-null    object                      
 2   pm10              545 non-null    float32                     
 3   pm2_5             545 non-null    float32                     
 4   carbon_monoxide   545 non-null    float32                     
 5   carbon_dioxide    495 non-null    float32                     
 6   dust              545 non-null    float32                     
 7   uv_index          545 non-null    float32                     
 8   nitrogen_dioxide  545 non-null    float32                     
dtypes: datetime64[ns, Asia/Jakarta](1), float32(7), object(1)
memory usage: 25.9+ KB


(600, 9)

In [36]:
df_air_quality_clean = df_air_quality.dropna(subset=['pm10', 'pm2_5'], how='all').copy()

df_air_quality_clean.tail()

,time,city_name,pm10,pm2_5,carbon_monoxide,carbon_dioxide,dust,uv_index,nitrogen_dioxide
584,2026-07-31 15:00:00+07:00,Surabaya,22.400000,18.299999,318.0,NaN,0.0,2.30,9.000000
585,2026-07-31 16:00:00+07:00,Surabaya,21.100000,17.299999,347.0,NaN,0.0,0.70,11.900000
586,2026-07-31 17:00:00+07:00,Surabaya,21.700001,18.100000,403.0,NaN,0.0,0.05,16.299999
587,2026-07-31 18:00:00+07:00,Surabaya,23.299999,20.100000,474.0,NaN,0.0,0.00,21.500000
588,2026-07-31 19:00:00+07:00,Surabaya,24.000000,20.799999,525.0,NaN,0.0,0.00,25.200001


In [38]:
import os
import sqlalchemy as sa

In [42]:
DB_PASSWORD = os.environ.get("SUPABASE_PASSWORD", "bikinporto123")

connection_url = sa.engine.URL.create(
    drivername="postgresql+psycopg2",
    username="postgres.jrmbgwfllqgaomytcbnb",
    password=DB_PASSWORD,
    host="aws-1-ap-south-1.pooler.supabase.com",
    port="6543",
    database="postgres"
)

engine = sa.create_engine(connection_url)

In [43]:
print("Uploading air quality data to Supabase...")

df_air_quality.to_sql(
    name='fact_air_quality',
    con=engine,
    schema="air_quality",
    if_exists='append',
    index=False,
    method='multi'
)

Uploading air quality data to Supabase...


600